# 04 — Statistical Testing (Mann–Whitney U)

**Section:** Evaluation layer (PROMETHEE II, CART baseline, statistical testing, GSEA)
**Depends on:** `03_promethee_ranking.ipynb` (`tests/fixtures/synthetic/reports/promethee_ranked_rules.csv`)
**Feeds into:** `05_gsea_enrichment.ipynb` (independent of this notebook's output, but next in sequence)

## Recommended reading before this notebook

- **Derrac et al., "A practical tutorial on the use of nonparametric statistical tests..."** —
  the reason this is Mann–Whitney U and not a paired/parametric test. GA and NSGA-II×baseline
  and ×SAM produce **independent** rule pools (different searches, not paired observations of
  the same unit), so this is the independent-samples nonparametric test, not Wilcoxon
  signed-rank (which needs paired samples) and not a t-test (which assumes normality this
  φ distribution has no reason to satisfy).
- **Orientation doc, Part C** — locks Mann–Whitney U specifically, over Wilcoxon, for exactly
  this reason. Recorded there as a settled decision, not an open question.
- **Methodology §4.6.2** — α = 0.05, two independent samples, on the φ (net outranking flow)
  distributions from the unified PROMETHEE run.

## What this notebook does

Loads notebook 03's φ-ranked table and runs `scipy.stats.mannwhitneyu` on two independent-
samples comparisons: **baseline vs SAM** (does SAM's generalisation-gap fix produce
higher-quality rules?) and **GA vs NSGA-II** (does the multi-objective search find better
rules than the scalarised single-objective one?). Both comparisons use the **same** unified
φ values from notebook 03 — nothing is re-ranked here, only compared.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

FIXTURE_DIR = Path("../tests/fixtures/synthetic")
REPORT_DIR = FIXTURE_DIR / "reports"

combined = pd.read_csv(REPORT_DIR / "promethee_ranked_rules.csv")
print(f"loaded {len(combined)} ranked rules")
combined.groupby(["extractor", "condition"]).size()


loaded 60 ranked rules


extractor  condition
ga         baseline     15
           sam          15
nsga       baseline     15
           sam          15
dtype: int64

## 1. Comparison groups

Two independent-samples comparisons, both drawn from the **same** φ column (no re-ranking):

1. **baseline vs SAM** — pools GA+NSGA-II within each condition (30 rules per side), testing
   whether the SAM intervention's smaller generalisation gap produces measurably different
   (by the paper's hypothesis, higher-φ) extracted rules. This is the paper's central claim.
2. **GA vs NSGA-II** — pools baseline+SAM within each extractor (30 rules per side), testing
   whether the two extraction methods differ in the rule quality they find, independent of
   which model condition they were run against.

**Note on synthetic data:** the fabricated rule pools (notebook 02) were built by uniform
random sampling over sparse chromosomes, not a real evolutionary search reflecting an actual
generalisation-gap effect — so a non-significant result here is *expected* on synthetic data
and is not informative about the real hypothesis. This notebook's job at this stage is to
prove the **test machinery** is correct (right test, right groups, right H1), not to produce
a meaningful verdict yet. That comes once real checkpoints and real GA/NSGA-II output replace
these fixtures.


In [2]:
def run_mannwhitney(group_a: pd.Series, group_b: pd.Series, label_a: str, label_b: str, alpha: float = 0.05) -> dict:
    u_stat, p_value = mannwhitneyu(group_a, group_b, alternative="two-sided")
    # rank-biserial correlation as an effect-size companion to the p-value -- U alone doesn't
    # convey magnitude, and a p-value without an effect size invites over-reading significance
    n_a, n_b = len(group_a), len(group_b)
    rank_biserial = 1 - (2 * u_stat) / (n_a * n_b)
    return {
        "comparison": f"{label_a} vs {label_b}",
        "n_a": n_a, "n_b": n_b,
        "median_a": float(np.median(group_a)), "median_b": float(np.median(group_b)),
        "U": float(u_stat), "p_value": float(p_value),
        "significant_at_0.05": bool(p_value < alpha),
        "rank_biserial_correlation": float(rank_biserial),
    }


results = []

baseline_phi = combined.loc[combined["condition"] == "baseline", "phi"]
sam_phi = combined.loc[combined["condition"] == "sam", "phi"]
results.append(run_mannwhitney(baseline_phi, sam_phi, "baseline", "sam"))

ga_phi = combined.loc[combined["extractor"] == "ga", "phi"]
nsga_phi = combined.loc[combined["extractor"] == "nsga", "phi"]
results.append(run_mannwhitney(ga_phi, nsga_phi, "ga", "nsga"))

results_df = pd.DataFrame(results)
results_df


,comparison,n_a,n_b,median_a,median_b,U,p_value,significant_at_0.05,rank_biserial_correlation
0,baseline vs sam,30,30,-0.053110,-0.034123,453.5,0.964616,False,-0.007778
1,ga vs nsga,30,30,-0.022968,-0.069935,556.0,0.118746,False,-0.235556


## 2. Interpreting the output

- **`U`**: Mann–Whitney U statistic (sum of rank-based wins for group A over group B).
- **`p_value`**: two-sided; below `alpha` (0.05) means the two φ distributions differ enough
  that random chance is an unlikely explanation.
- **`rank_biserial_correlation`**: effect size in [-1, 1] — 0 means the two groups' rank
  distributions are indistinguishable; ±1 means complete separation. Reported alongside the
  p-value per good-practice nonparametric reporting (Derrac et al.), since a large N can make
  a trivially small effect "significant" and a small N can hide a real one.

Both comparisons below are expected to be non-significant on this fabricated data (see §1) —
confirming the test runs correctly and produces sane numbers is the goal here, not a verdict.


In [3]:
for r in results:
    verdict = "SIGNIFICANT" if r["significant_at_0.05"] else "not significant"
    print(f"{r['comparison']}: U={r['U']:.1f}, p={r['p_value']:.4f} ({verdict}), "
          f"medians {r['median_a']:.3f} vs {r['median_b']:.3f}, "
          f"rank-biserial r={r['rank_biserial_correlation']:.3f}")


baseline vs sam: U=453.5, p=0.9646 (not significant), medians -0.053 vs -0.034, rank-biserial r=-0.008
ga vs nsga: U=556.0, p=0.1187 (not significant), medians -0.023 vs -0.070, rank-biserial r=-0.236


## 3. Save outputs

Same placement logic as notebook 03 (§6 there): this is evaluation output computed over
fabricated rule pools, so it stays under `tests/fixtures/synthetic/reports/`, not
`artifacts/reports/`, until real checkpoints/rule pools replace the fixtures.


In [4]:
results_df.to_csv(REPORT_DIR / "mannwhitney_results.csv", index=False)
print(f"wrote {len(results_df)}-row Mann-Whitney results -> {REPORT_DIR / 'mannwhitney_results.csv'}")


wrote 2-row Mann-Whitney results -> ../tests/fixtures/synthetic/reports/mannwhitney_results.csv


## Known gaps, to address when real data arrives

- Results are **not yet meaningful** — synthetic rule pools are uniform-random fabrications,
  not real evolutionary search output, so non-significance here proves nothing about the real
  hypothesis (see §1). Re-run against real GA/NSGA-II pools once Melisa's and Sakhile's
  branches land.
- Comparison groupings (baseline-vs-SAM pooling both extractors; GA-vs-NSGA-II pooling both
  conditions) are a reasonable default but haven't been confirmed with the team — flag if a
  four-way pairwise comparison (e.g. `ga_baseline` vs `nsga_sam` specifically) is wanted
  instead or in addition.
- Per-drug repetition: once the drug shortlist is fixed (Part F, orientation doc), this test
  runs once per drug — the comparison groups above generalise directly, just re-run per drug's
  `promethee_ranked_rules.csv`.

## Next up

- **`05_gsea_enrichment.ipynb`** — pathway over-representation (GSEApy) on the top-ranked
  rules' `genes` fields, independent of this notebook's statistical output.
- **`06_cart_baseline.ipynb`** — once built, its rules flow back through notebook 03's unified
  PROMETHEE run, and this notebook's GA-vs-NSGA-II pattern extends naturally to an
  evolutionary-vs-CART comparison.
